In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from arch import arch_model
from arch.univariate import ARX
from scipy.stats import norm

### Import data

In [2]:
path = os.path.abspath('E:/RA/Geert/task1.py')
dir_path = os.path.dirname(path)
os.chdir(dir_path)
excel_file = pd.ExcelFile('Aggregate_CPI_inflation_20230513.xls')
sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
#quarterly and monthly aggregate CPI data (deseasonalized). The full sample is 1947-2022
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))
data_month.index = pd.to_datetime(data_month[['Year', 'Month']].assign(day=1))
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
sample_data = data_quarter[data_quarter['Year']>1969]

In [6]:
sample_data['Inflation_lag_1'] =  sample_data['Inflation'].shift(1)
sample_data['Inflation_lag_2'] =  sample_data['Inflation'].shift(2)
sample_data['Forecasted_inflation_lag_1'] =  sample_data['Forecasted inflation'].shift(1)
sample_data = sample_data.dropna()

### Mean model and Variance model are jointly estimated. 

In [7]:
def GarchFamilyResults(Y,X=None,mean='Zero',lags=None, cov = 'robust'):
    name_dict = {'normal':'Normal Distribution','studentst':'Standard Student t distribution','skewstudent':'Skew Student t distribution','generalized error':'Generalized Error distribution'}
    options = {'maxiter': 1000}
    
    print('\n\nFollowing results are using Mixture of 2 Normals Distribution')
    dparams=[0.3,0.1,1]
    sparch11 = SPARCH(Y=Y,dparams=dparams,X=X,mean=mean, vol='GARCH',p=1,q=1,lags=lags)
    sparch21 = SPARCH(Y=Y,dparams=dparams,X=X,mean=mean, vol='GARCH',p=2,q=1,lags=lags)
    sparch12 = SPARCH(Y=Y,dparams=dparams,X=X,mean=mean, vol='GARCH',p=1,q=2,lags=lags)   
    sparch22 = SPARCH(Y=Y,dparams=dparams,X=X,mean=mean, vol='GARCH',p=2,q=2,lags=lags)
    print('\nSPARCH(p,q) model')
    print('\t \t AIC: \t \t \t BIC',
          '\nGARCH(1,1): ',sparch11.aic,'\t',sparch11.bic,
          '\nGARCH(2,1): ',sparch21.aic,'\t',sparch21.bic,
          '\nGARCH(1,2): ',sparch12.aic,'\t',sparch12.bic,
          '\nGARCH(2,2): ',sparch22.aic,'\t',sparch22.bic,)
    
 
    sparch11 = SPARCH(Y=Y,dparams=dparams,X=X,mean=mean, vol='GARCH',p=1,o=1,q=1,lags=lags)
    sparch21 = SPARCH(Y=Y,dparams=dparams,X=X,mean=mean, vol='GARCH',p=2,o=1,q=1,lags=lags)
    sparch12 = SPARCH(Y=Y,dparams=dparams,X=X,mean=mean, vol='GARCH',p=1,o=1,q=2,lags=lags)   
    sparch22 = SPARCH(Y=Y,dparams=dparams,X=X,mean=mean, vol='GARCH',p=2,o=2,q=2,lags=lags)
    print('\nSPARCH(p,q) model')
    print('\t \t AIC: \t \t \t BIC',
          '\nGJR-GARCH(1,1): ',sparch11.aic,'\t',sparch11.bic,
          '\nGJR-GARCH(2,1): ',sparch21.aic,'\t',sparch21.bic,
          '\nGJR-GARCH(1,2): ',sparch12.aic,'\t',sparch12.bic,
          '\nGJR-GARCH(2,2): ',sparch22.aic,'\t',sparch22.bic,) 

In [75]:
def panel_aic():
    rest_aic = {
    'GJR':{ 'Best Mean Model': [],
     'Best Volatility Process': [],
     'AIC': []} ,
    'Egarch':{ 'Best Mean Model': [],
     'Best Volatility Process': [],
    'AIC': [], }
    }  
    rest_bic = {
    'GJR':{ 'Best Mean Model': [],
     'Best Volatility Process': [],
     'BIC': []} ,
    'Egarch':{ 'Best Mean Model': [],
     'Best Volatility Process': [],
    'BIC': [], }
    }

    options = {'maxiter': 1000}
    mean_fc = { 'y': sample_data['Inflation shock'], 'x': None, 'mean': 'Zero', 'lags': None, }
    mean_11 = { 'y': sample_data['Inflation'],'x': sample_data['Forecasted inflation'], 'mean': 'ARX', 'lags': [1],}          
    mean_21 = { 'y': sample_data['Inflation'],'x': sample_data['Forecasted inflation'],'mean': 'ARX', 'lags': [1,2], }          
    mean_22 = {'y': sample_data['Inflation'],'x': sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']],'mean': 'ARX','lags': [1,2], }
    mean_model = [mean_fc,mean_11,mean_21,mean_22]
    name_dict = ['normal','studentst','skewstudent','generalized error']
    
    for dist in name_dict:
        #aic panel 
        gjr_rest=[]
        egarch_rest=[]
        for mean,_name in zip(mean_model,['FC','(1,1)','(2,1)','(2,2)']):
            gjr_garch11 = arch_model(**mean,p=1, o=1,q=1,dist=dist).fit(disp='off',options=options)
            gjr_rest.append(( _name ,'(1,1)',gjr_garch11.aic))
            gjr_garch12 = arch_model(**mean,p=1, o=1,q=2,dist=dist).fit(disp='off',options=options)
            gjr_rest.append(( _name ,'(1,2)',gjr_garch12.aic))
            gjr_garch21 = arch_model(**mean,p=2, o=2,q=1,dist=dist).fit(disp='off',options=options)
            gjr_rest.append(( _name ,'(2,1)',gjr_garch21.aic))
            gjr_garch22 = arch_model(**mean,p=2, o=2,q=2,dist=dist).fit(disp='off',options=options)
            gjr_rest.append(( _name ,'(2,2)',gjr_garch22.aic))
                    
            egarch11 = arch_model(**mean,p=1, o=1, q=1, vol='EGARCH',dist=dist).fit(disp='off',options=options)
            egarch_rest.append(( _name ,'(1,1)',egarch11.aic))
            egarch12 = arch_model(**mean,p=1, o=1, q=2, vol='EGARCH',dist=dist).fit(disp='off',options=options)
            egarch_rest.append(( _name ,'(1,2)',egarch11.aic))
            egarch21 = arch_model(**mean,p=2, o=2, q=1, vol='EGARCH',dist=dist).fit(disp='off',options=options)
            egarch_rest.append(( _name ,'(2,1)',egarch11.aic))
            egarch22 = arch_model(**mean,p=2, o=2, q=2, vol='EGARCH',dist=dist).fit(disp='off',options=options)
            egarch_rest.append(( _name ,'(2,2)',egarch11.aic))
            
        best_gir,best_egarch = min(gjr_rest, key=lambda x: x[2]),min(egarch_rest, key=lambda x: x[2])
        rest_aic['GJR'][ 'Best Mean Model'].append(best_gir[0])
        rest_aic['GJR'][ 'Best Volatility Process'].append(best_gir[1])
        rest_aic['GJR'][ 'AIC'].append(best_gir[2])
        rest_aic['Egarch']['Best Mean Model'].append(best_egarch[0])
        rest_aic['Egarch'][ 'Best Volatility Process'].append(best_egarch[1])
        rest_aic['Egarch'][ 'AIC'].append(best_egarch[2])
        
    gjr_rest=[]
    egarch_rest=[]
    for mean,_name in zip(mean_model,['FC','(1,1)','(2,1)','(2,2)']):  
        sparch11 = SPARCH(**mean, vol='GARCH',p=1,o=1,q=1)
        gjr_rest.append(( _name ,'(1,1)',sparch11.aic))
        sparch21 = SPARCH(**mean, vol='GARCH',p=2,o=2,q=1)
        gjr_rest.append(( _name ,'(2,1)',sparch21.aic))
        sparch12 = SPARCH(**mean, vol='GARCH',p=1,o=1,q=2)
        gjr_rest.append(( _name ,'(1,2)',sparch12.aic))
        sparch22 = SPARCH(**mean, vol='GARCH',p=2,o=2,q=2)
        gjr_rest.append(( _name ,'(2,2)',sparch22.aic))
        
        sparch11 = SPARCH(**mean, vol='EGARCH',p=1,o=1,q=1)
        egarch_rest.append(( _name ,'(1,1)',sparch11.aic))
        sparch21 = SPARCH(**mean, vol='EGARCH',p=2,o=2,q=1)
        egarch_rest.append(( _name ,'(2,1)',sparch21.aic))
        sparch12 = SPARCH(**mean, vol='EGARCH',p=1,o=1,q=2)
        egarch_rest.append(( _name ,'(1,2)',sparch12.aic))
        sparch22 = SPARCH(**mean, vol='EGARCH',p=2,o=2,q=2)   
        egarch_rest.append(( _name ,'(2,2)',sparch22.aic))
        
    best_gir,best_egarch = min(gjr_rest, key=lambda x: x[2]),min(egarch_rest, key=lambda x: x[2])
    rest_aic['GJR'][ 'Best Mean Model'].append(best_gir[0])
    rest_aic['GJR'][ 'Best Volatility Process'].append(best_gir[1])
    rest_aic['GJR'][ 'AIC'].append(best_gir[2])
    rest_aic['Egarch']['Best Mean Model'].append(best_egarch[0])
    rest_aic['Egarch'][ 'Best Volatility Process'].append(best_egarch[1])
    rest_aic['Egarch'][ 'AIC'].append(best_egarch[2])   
        
    index = ['Normal', 't', 'Skew t', 'GED','Mix of Normal']
    df_gjr = pd.DataFrame(rest_aic['GJR'], index=index)
    df_gjr.columns = pd.MultiIndex.from_product([['GJR'], df_gjr.columns])
    df_egarch = pd.DataFrame(rest_aic['Egarch'], index=index)
    df_egarch.columns = pd.MultiIndex.from_product([['Egarch'], df_egarch.columns])
    return pd.concat([df_gjr, df_egarch], axis=1)


In [76]:
aic = panel_aic()

D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outs

D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bo

D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outs

D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bo

In [77]:
aic.to_excel(r'E:\RA\Geert\aic.xlsx')

## Panel of AIC & BIC

In [78]:
def panel_bic():
    rest_bic = {
    'GJR':{ 'Best Mean Model': [],
     'Best Volatility Process': [],
     'BIC': []} ,
    'Egarch':{ 'Best Mean Model': [],
     'Best Volatility Process': [],
    'BIC': [], }
    }

    options = {'maxiter': 1000}
    mean_fc = { 'y': sample_data['Inflation shock'], 'x': None, 'mean': 'Zero', 'lags': None, }
    mean_11 = { 'y': sample_data['Inflation'],'x': sample_data['Forecasted inflation'], 'mean': 'ARX', 'lags': [1],}          
    mean_21 = { 'y': sample_data['Inflation'],'x': sample_data['Forecasted inflation'],'mean': 'ARX', 'lags': [1,2], }          
    mean_22 = {'y': sample_data['Inflation'],'x': sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']],'mean': 'ARX','lags': [1,2], }
    mean_model = [mean_fc,mean_11,mean_21,mean_22]
    name_dict = ['normal','studentst','skewstudent','generalized error']
    
    for dist in name_dict:
        #aic panel 
        gjr_rest=[]
        egarch_rest=[]
        for mean,_name in zip(mean_model,['FC','(1,1)','(2,1)','(2,2)']):
            gjr_garch11 = arch_model(**mean,p=1, o=1,q=1,dist=dist).fit(disp='off',options=options)
            gjr_rest.append(( _name ,'(1,1)',gjr_garch11.bic))
            gjr_garch12 = arch_model(**mean,p=1, o=1,q=2,dist=dist).fit(disp='off',options=options)
            gjr_rest.append(( _name ,'(1,2)',gjr_garch12.bic))
            gjr_garch21 = arch_model(**mean,p=2, o=2,q=1,dist=dist).fit(disp='off',options=options)
            gjr_rest.append(( _name ,'(2,1)',gjr_garch21.bic))
            gjr_garch22 = arch_model(**mean,p=2, o=2,q=2,dist=dist).fit(disp='off',options=options)
            gjr_rest.append(( _name ,'(2,2)',gjr_garch22.bic))
                    
            egarch11 = arch_model(**mean,p=1, o=1, q=1, vol='EGARCH',dist=dist).fit(disp='off',options=options)
            egarch_rest.append(( _name ,'(1,1)',egarch11.bic))
            egarch12 = arch_model(**mean,p=1, o=1, q=2, vol='EGARCH',dist=dist).fit(disp='off',options=options)
            egarch_rest.append(( _name ,'(1,2)',egarch11.bic))
            egarch21 = arch_model(**mean,p=2, o=2, q=1, vol='EGARCH',dist=dist).fit(disp='off',options=options)
            egarch_rest.append(( _name ,'(2,1)',egarch11.bic))
            egarch22 = arch_model(**mean,p=2, o=2, q=2, vol='EGARCH',dist=dist).fit(disp='off',options=options)
            egarch_rest.append(( _name ,'(2,2)',egarch11.bic))
            
        best_gir,best_egarch = min(gjr_rest, key=lambda x: x[2]),min(egarch_rest, key=lambda x: x[2])
        rest_bic['GJR'][ 'Best Mean Model'].append(best_gir[0])
        rest_bic['GJR'][ 'Best Volatility Process'].append(best_gir[1])
        rest_bic['GJR'][ 'BIC'].append(best_gir[2])
        rest_bic['Egarch']['Best Mean Model'].append(best_egarch[0])
        rest_bic['Egarch'][ 'Best Volatility Process'].append(best_egarch[1])
        rest_bic['Egarch'][ 'BIC'].append(best_egarch[2])
        
    gjr_rest=[]
    egarch_rest=[]
    for mean,_name in zip(mean_model,['FC','(1,1)','(2,1)','(2,2)']):  
        sparch11 = SPARCH(**mean, vol='GARCH',p=1,o=1,q=1)
        gjr_rest.append(( _name ,'(1,1)',sparch11.bic))
        sparch21 = SPARCH(**mean, vol='GARCH',p=2,o=2,q=1)
        gjr_rest.append(( _name ,'(2,1)',sparch21.bic))
        sparch12 = SPARCH(**mean, vol='GARCH',p=1,o=1,q=2)
        gjr_rest.append(( _name ,'(1,2)',sparch12.bic))
        sparch22 = SPARCH(**mean, vol='GARCH',p=2,o=2,q=2)
        gjr_rest.append(( _name ,'(2,2)',sparch22.bic))
        
        sparch11 = SPARCH(**mean, vol='EGARCH',p=1,o=1,q=1)
        egarch_rest.append(( _name ,'(1,1)',sparch11.bic))
        sparch21 = SPARCH(**mean, vol='EGARCH',p=2,o=2,q=1)
        egarch_rest.append(( _name ,'(2,1)',sparch21.bic))
        sparch12 = SPARCH(**mean, vol='EGARCH',p=1,o=1,q=2)
        egarch_rest.append(( _name ,'(1,2)',sparch12.bic))
        sparch22 = SPARCH(**mean, vol='EGARCH',p=2,o=2,q=2)   
        egarch_rest.append(( _name ,'(2,2)',sparch22.bic))
        
    best_gir,best_egarch = min(gjr_rest, key=lambda x: x[2]),min(egarch_rest, key=lambda x: x[2])
    rest_bic['GJR'][ 'Best Mean Model'].append(best_gir[0])
    rest_bic['GJR'][ 'Best Volatility Process'].append(best_gir[1])
    rest_bic['GJR'][ 'BIC'].append(best_gir[2])
    rest_bic['Egarch']['Best Mean Model'].append(best_egarch[0])
    rest_bic['Egarch'][ 'Best Volatility Process'].append(best_egarch[1])
    rest_bic['Egarch'][ 'BIC'].append(best_egarch[2])   
        
    index = ['Normal', 't', 'Skew t', 'GED','Mix of Normal']
    df_gjr = pd.DataFrame(rest_bic['GJR'], index=index)
    df_gjr.columns = pd.MultiIndex.from_product([['GJR'], df_gjr.columns])
    df_egarch = pd.DataFrame(rest_bic['Egarch'], index=index)
    df_egarch.columns = pd.MultiIndex.from_product([['Egarch'], df_egarch.columns])
    return pd.concat([df_gjr, df_egarch], axis=1)


In [79]:
bic = panel_bic()

D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bo

D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the mod

D:\anaconda\lib\site-packages\arch\univariate\base.py:704: StartingValueWarning: Starting values do not satisfy the parameter constraints in the model.  The
provided starting values will be ignored.

  warnings.warn(starting_value_warning, StartingValueWarning)
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outs

D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  warnings.warn("Values in x were outside bounds during a "
D:\anaconda\lib\site-packages\scipy\optimize\optimize.py:282: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bo

In [80]:
bic.to_excel(r'E:\RA\Geert\bic.xlsx')

In [63]:
from arch.univariate.distribution import Distribution
from abc import ABCMeta, abstractmethod
from collections.abc import Sequence
from typing import Callable
import warnings

from numpy import (
    abs,    array,    asarray,    empty,    exp,    int64,    
    integer,    isscalar,    log,    nan,    ndarray, exp,
    ones_like,    pi,    sign,    sqrt,    sum, random)
from numpy.random import Generator, RandomState, default_rng
from scipy.special import comb, gamma, gammainc, gammaincc, gammaln
import scipy.stats as stats
from scipy.optimize import bisect

from arch.typing import ArrayLike, ArrayLike1D, Float64Array
from arch.utility.array import AbstractDocStringInheritor, ensure1d

class MixNormal(Distribution, metaclass=AbstractDocStringInheritor):
    """
    Mixture of two Normal distributions for use with SPARCH model

    Parameters
    ----------
    random_state : RandomState, optional
        .. deprecated:: 5.0

           random_state is deprecated. Use seed instead.

    seed : {int, Generator, RandomState}, optional
        Random number generator instance or int to use. Set to ensure
        reproducibility. If using an int, the argument is passed to
        ``np.random.default_rng``.  If not provided, ``default_rng``
        is used with system-provided entropy.
    """

    def __init__(
        self,
        random_state: RandomState | None = None,
        *,
        seed: None | int | RandomState | Generator = None,
    ) -> None:
        super().__init__(random_state=random_state, seed=seed)
        self._name = "Mixture of two Normal distributions"
        self.num_params: int = 3  

    def constraints(self) -> tuple[Float64Array, Float64Array]:
        return empty(0), empty(0)
        #return array([[1, 0, 0], [-1, 0, 0], [0, 1,0], [0, -1,0] , [0,0,1],[0,0,-1]]), array([0.05, 0.95, -20,20,0.2, 10])

    def bounds(self, resids: Float64Array) -> list[tuple[float, float]]:
        """
        Bounds of parameters:
        p1: (0,1)
        u1: (-10,10)
        sigma_1:(0.001,10)
        """
        return [(0.0001, 0.9999),(-10000,10000),(0.0001, 10000)]

    def loglikelihood(
        self,
        parameters: Sequence[float] | ArrayLike1D,
        resids: ArrayLike,
        sigma2: ArrayLike,
        individual: bool = False,
    ) -> float | Float64Array:
        r"""Computes the log-likelihood of assuming residuals are mixture normally
        distributed, conditional on the variance

        Parameters
        ----------
        parameters : ndarray
            Parameters of the first normal distribution: p1,u1,sigma1. Second one can be calculated by restrictions.
        resids  : ndarray
            The residuals to use in the log-likelihood calculation
        sigma2 : ndarray
            Conditional variances of resids
        individual : bool, optional
            Flag indicating whether to return the vector of individual log
            likelihoods (True) or the sum (False)

        Returns
        -------
        ll : float
            The log-likelihood

        Notes
        -----
        The log-likelihood of a single data point x is

        .. math::

            \ln f\left(x\right)=
            \ln \frac{1}{\sqrt{h_t}} \left[
                 p_1  \frac{1}{\sqrt{ 2\pi\sigma_1^2} } exp\{ -\frac{(x-\mu_1)^2}{2\sigma_1^2} \}
                +(1-p_1)\frac{1}{\sqrt{ 2\pi\sigma_2^2} } exp\{ -\frac{(x-\mu_2)^2}{2\sigma_2^2} \} \right]

        """
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        z = resids/sqrt(sigma2)
        warnings.filterwarnings("ignore")
        lls =-1/2 *log(sigma2) +  log(   p1/(sqrt(2*pi*sigma_1_2)) *  exp(-( z-u1)**2/(2*sigma_1_2)) 
                                     +   p2/(sqrt(2*pi*sigma_2_2)) *  exp(-( z-u2)**2/(2*sigma_2_2))   )
        warnings.filterwarnings("default")
          
        if individual:
            return lls
        else:
            return sum(lls)

    def starting_values(self, std_resid: Float64Array) -> Float64Array:
        """
        Starting values of parameters
        """
        #gmm = GaussianMixture(n_components=2).fit(std_resid.reshape(-1,1))
        #return array([gmm.weights_[0],gmm.means_[0][0], gmm.covariances_[0][0][0] ])
        return array([0.3, 0.1,1])
    
    def _simulator(self, size: int | tuple[int, ...]) -> Float64Array:
        assert self._parameters is not None
        p1, u1, sigma_1_2 = self._parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        Z = np.random.binomial(n=1, p=p1, size=size)
        return sqrt(sigma_1_2)**Z*sqrt(sigma_2_2)**(1-Z)*self._generator.standard_normal(size) + u1*Z+ u2*(1-Z)

    def simulate(
        self, parameters: int | float | Sequence[float | int] | ArrayLike1D
    ) -> Callable[[int | tuple[int, ...]], Float64Array]:
        parameters = ensure1d(parameters, "parameters", False)
        self._parameters = asarray(parameters, dtype=float)
        return self._simulator

    def parameter_names(self) -> list[str]:
        return ['p_1','mu_1','sigma_1^2']

    def cdf(
        self,
        resids: Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        return p1*stats.norm.cdf(asarray((resids-u1)/sqrt(sigma_1_2))  ) + p2*stats.norm.cdf(asarray((resids-u2)/sqrt(sigma_2_2)))

    def ppf(
        self,
        pits: float | Sequence[float] | ArrayLike1D,
        parameters: None | Sequence[float] | ArrayLike1D = None,
    ) -> Float64Array:
        self._check_constraints(parameters)
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        scalar = isscalar(pits)
        if scalar:
            pits = array([pits])
        else:
            pits = asarray(pits)
            
        def inverse_cdf(cdf, target_p, lower_bound=-100, upper_bound=100):
            def root_func(x):
                return cdf(x,parameters) - target_p
            return bisect(root_func, lower_bound, upper_bound)     
        
        ppf = inverse_cdf(self.cdf, pits, lower_bound=-100, upper_bound=100)

        if scalar:
            return ppf[0]
        else:
            return ppf

    def moment(
        self, n: int, parameters: None | Sequence[float] | ArrayLike1D = None
    ) -> float:
        if n < 0:
            return nan
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        moment1 = stats.norm.moment(n,loc=u1,scale=sqrt(sigma_1_2))
        moment2 = stats.norm.moment(n,loc=u2,scale=sqrt(sigma_2_2))
        return p1 * moment1 + p2 * moment2

    def partial_moment(
        self,
        n: int,
        z: float = 0.0,
        parameters: None | Sequence[float] | ArrayLike1D = None,
        num_samples=1000,
    ) -> float:
        
        parameters = asarray(parameters, dtype=float)
        p1, u1, sigma_1_2 = parameters
        p2 = 1- p1
        u2 = p1/(p1-1) *u1
        sigma_2_2 = (1-p2*u2*u2-p1*(sigma_1_2+u2*u2))/(p2)
        
        if n < 0:
            return nan
        elif n == 0:
            return cdf(z,parameters)
        elif n==1:
            return -p1*stats.norm.pdf(z,loc=u1,scale=sqrt(sigma_1_2))  -p2*stats.norm.pdf(z,loc=u2,scale=sqrt(sigma_2_2))
        else:
            -(z ** (n - 1)) * (p1*stats.norm.pdf(z,loc=u1,scale=sqrt(sigma_1_2))+p2*stats.norm.pdf(z,loc=u2,scale=sqrt(sigma_2_2))) 
            + (n - 1) * self.partial_moment(  n - 2, z, parameters  )
def SPARCH(y,x,mean, p,o,q,lags,vol='GARCH',cov = 'robust'):
    '''
    step 1: run a garch with student t distribution; estimate gaussian mixture parameters from garch residuals
    step 2: run a sparch using garch parameters
    '''
    #run GARCH with student t distribuion as banchmark
    dparams=[0.9,0,1]
    garht = arch_model(y=y,x=x, mean=mean, vol=vol,p=p, o=o,q=q,lags=lags,dist='studentst').fit(disp='off',cov_type=cov)
    params = array(garht.params) +   0.1*array(garht.std_err)*np.random.normal(size=garht.std_err.shape)
    starting_values = np.concatenate( (np.array(params)[:-1],dparams )   )
    constraints = {'type': 'ineq',
                'fun': lambda x: array([ 0.9999 - x[-3] * (x[-2]**2 + x[-1]) - (1 - x[-3]) * (-x[-3] * x[-2] / (1 - x[-3]))**2])} 
    options={'maxiter': 1000,'constraints':constraints}
    sparch = arch_model(y=y,x=x, mean=mean, vol=vol,p=p, o=o,q=q,lags=lags)
    sparch.distribution = MixNormal()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        MixNormalres=sparch.fit(disp='off',cov_type=cov,options=options,starting_values=starting_values )
    return MixNormalres